# Task 1 MongoDB Setup

Explanation regarding collection setup and relationships are provided in detail in [mongo_setup_explanation.md](mongo_setup_explanation.md)

In [1]:
from pymongo import MongoClient
from pathlib import Path
import pandas as pd

client = MongoClient("mongodb://mongodb:27017/") # net stop mongodb if local is running
db = client["fit3182_a2"]

data_path = Path("..") / "data"

def db_setup():
    """
    Initializes and configures the MongoDB structure for the AWAS system.
    
    Will drop existing collections to ensure a clean state for each collection, then creating the vehicles,
    cameras, and violations collections with appropriate indexes to optimize query performance and ensure data integrity.
    
    Args:
        None
    Returns:
        None
    """
    
    # drop existing collections
    db.vehicles.drop()
    db.cameras.drop()
    db.violations.drop()
    
    # Vehicle collection setup
    vehicle_collection = db["vehicles"]
    
    vehicle_collection.create_index([("car_plate", 1)], unique=True)
    
    # Camera collection setup
    camera_collection = db["cameras"]
    camera_collection.create_index([("camera_id", 1)], unique=True)
    camera_collection.create_index([("location", "2dsphere")])
    
    # Violations collection setup
    violation_collection = db["violations"]
    violation_collection.create_index([("car_plate", 1), ("date", -1)])

def populate_db():
    """
    Populates the vehicles and cameras collections with static reference data from the provided CSV files.
    
    Implements upsert logic for vehicles to handle duplicate vehicles, only storing vehicles with the most recent
    registration date.
    
    Args:
        None
    Returns:
        None
    """
    
    vehicle_csv = pd.read_csv(data_path / "vehicle.csv")
    camera_csv = pd.read_csv(data_path / "camera.csv")
    
    # handle vehicle insertion
    for _, row in vehicle_csv.iterrows():
        data_to_insert = {
            "car_plate": row['car_plate'],
            "owner_name": row['owner_name'],
            "owner_addr": row['owner_addr'],
            "vehicle_type": row['vechicle_type'],  # Note: typo in source CSV
            "registration_date": pd.to_datetime(row['registration_date']),
            "created_at": pd.Timestamp.utcnow()
        }
        
        # Duplication handling: check if a record with the same car_plate exists
        vehicle_exists = db.vehicles.find_one({"car_plate": row['car_plate']})
        
        if vehicle_exists:
            vehicle_exists_date = vehicle_exists.get("registration_date", pd.Timestamp.min)
            # replace old record if new one has a more recent registration_date
            if pd.to_datetime(row['registration_date']) > vehicle_exists_date: 
                db.vehicles.update_one(
                    {"car_plate": row['car_plate']},
                    {"$set": {**data_to_insert, "updated_at": pd.Timestamp.utcnow()}}
                )
        else:
            db.vehicles.insert_one(data_to_insert)
    
    # handle camera insertion
    for _, row in camera_csv.iterrows():
        data_to_insert = {
            "camera_id": int(row['camera_id']),
            "location": {
                "type": "Point",
                "coordinates": [float(row['longitude']), float(row['latitude'])]
            },
            "position": float(row['position']),
            "speed_limit": int(row['speed_limit']),
            "created_at": pd.Timestamp.utcnow()
        }
    
        db.cameras.update_one(
            {"camera_id": int(row['camera_id'])},
            {"$setOnInsert": data_to_insert},
            upsert=True
        )

print("Setting up database collections and indexes...")
db_setup()
print("Database collections and indexes created.")
populate_db()
print("Initial data population complete.")
print("Database setup and initial population complete.")

Setting up database collections and indexes...
Database collections and indexes created.
Initial data population complete.
Database setup and initial population complete.


# Task 2 Streaming Application

This notebook implements the streaming pipeline for the AWAS traffic mnonitoring system, which covers Kafka stream ingestion, stream-to-static joins with camera metadata, violation detection (average and instantaneous), and MongoDB sink integration.

**Pipeline:**
1. Ingesting camera event streams from Kafka through Producers A, B, C which each reads data from camera-events-A, camera-events-B, and camera-events-C respectively.
2. Joining each stream with camera metadata to retrieve speed limit, position, and coordinates
3. Detecting instantaneous violations, vehicles whose recorded speed exceeds the camera's speed limit will be flagged
4. Detecting average speed violations, by joniing entry and exit events then calculating the average speed, we can check if the speed limit has been breached (Road A->B and Road B->C)
5. Pushing all violations to MongoDB, grouped by car plate and violation date

**Assumptions**
**Checkpointing Strategy:** Persistent checkpoints (`checkpointLocation`) are intentionally omitted for this academic implementation. Since the pipeline runs in a local Jupyter environment and is restarted fresh per demonstration, Spark's default temporary checkpoints suffice. This prevents stale state conflicts between runs and simplifies local testing. In a production AWAS deployment, checkpoints would be mandatory to guarantee driver fault tolerance, exactly-once offset commits, and state recovery after failures.

### Environment Setup + Spark


In [2]:
import glob
import os
import shutil
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date, sin, cos, sqrt, atan2, radians
)
from pyspark.sql.types import *
from datetime import datetime
import time

HOST_IP = "192.168.64.1"
MONGO_URI = "mongodb://mongodb:27017/"
MONGO_DB = "fit3182_a2"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

Debug: SparkSession has been created successfully.


# Watermark Design & Justification

### 1. Dataset Lateness Analysis (Data Mining)
Before configuring Spark Structured Streaming watermarks, we performed an analysis of out-of-order event arrivals in the provided CSV datasets. For each camera stream, we computed the maximum observed lateness in [delay_calculation.ipynb](delay_calculation.ipynb), the calculation was done as such:
- Maintained a running cumulative maximum of event timestamps (`cummax`)
- Calculated the difference between each event's timestamp and the maximum timestamp seen so far
- Identified the largest negative gap, which represents the worst case out-of-order arrival in the historical data

**Calculated Maximum Lateness per Stream: (Rounded up to the nearest second)**
| Stream | Max Observed Lateness |
|--------|----------------------|
| Camera A | `6 seconds` |
| Camera B | `647 seconds` (~10.8 min) |
| Camera C | `1834 seconds` (~30.5 min) |

### 2. Buffer Addition for Future Data & Operational Safety Margin
While the historical analysis provides a baseline for watermark configuration, deploying a streaming application in production requires accounting for unseen future patterns and operational variability. Therefore, we apply a safety buffer to each stream's observed maximum lateness:
- Camera A's ~6s lateness gap was rounded up to the nearest minute (60 seconds), which only has negligible overhead, especially since A is joined with B where B has a significantly larger watermark which will end up becoming the global watermark for the join.
- Camera B and C received a 10% safety margin (rounded up) to accomodate minor timing variations while avoiding excessive state retention.
- This watermarking approach ensures no valid event pairs are prematurely dropped while keeping Spark's state management under manageable levels. 

### 3. Maintenance and Watermark Readjusment
Watermark configuration will not be static in this system, there should be a periodic maintenance protocol that helps in ensuring correctness and efficiency. To do so, several things can be done:
1. Monitoring dropped records:
Our current streaming job logs all records evicted due to watermark expiry, if we collect this data and we see a sustained increase in dropped pairs for a given stream, we can infer that the current watermark may be too aggressive and may need readjustments.
2. Scheduled re-analysis:
After a certain period of time, we can run the maximum observed lateness analysis on newly ingested data, ensuring that our watermark stays up to date with new data.
3. Monitoring strategies:
A monitoring strategy can always be implemented if the number of dropped pairs increase substantially over a period if time, alerting the responsible engineering team behind the system when it increases by a certain percentage.

This maintenance strategy ensures that the watermark configuration evolves with the data.

### 4. Assumptions and Trade-offs
**Scalability & State Retention Constraints**
This watermark strategy assumes bounded event volumes and moderate throughput. In Spark Structured Streaming, watermarks dictate how long unmatched entry events are retained in the state store. With our current buffer, every unmatched entry is held until the global watermark expiry. As data volume scales (e.g., city-wide deployment, higher vehicle density, or sustained network latency), state retention grows linearly. This can lead to increased heap/RocksDB memory pressure, larger checkpoint files, longer state compaction cycles, and micro-batch processing delays. Under extreme scale, this approach risks `OutOfMemoryError` or backpressure if infrastructure is not proportionally scaled.

**Correctness vs. Resource Efficiency**
Our configuration deliberately prioritises **correctness over resource efficiency**. By retaining state longer, we minimise false negatives (dropping valid but delayed entry/exit pairs) and ensure all legitimate violations are captured. This comes at a computational cost: higher memory footprint, increased garbage collection overhead, and slightly longer batch completion times. A more aggressive (smaller) watermark would aggressively evict state, reducing resource consumption and improving throughput/latency, but at the risk of prematurely dropping valid late-arriving events. This represents a fundamental streaming systems trade-off: guaranteeing detection completeness vs. optimising for bounded state and cost-efficient processing.

**When a More Aggressive Watermark Becomes Necessary**
If event throughput increases significantly or infrastructure costs become prohibitive, we would transition to a tighter watermark strategy. Production alternatives include:
- Using latency percentiles (e.g., P95/P99) instead of maximum observed gaps
- Implementing dynamic watermark adjustment based on real-time state store metrics
- Adopting a two-tier architecture: low-latency stream for real-time enforcement + batch reconciliation for late events
In practice, watermark tuning is iterative, monitored via Spark UI state metrics, and balanced against operational SLAs for violation detection latency and resource budgets.


In [3]:
import math
# Set watermark per camera
camera_a_lateness_data = 6
camera_b_lateness_data = 647
camera_c_lateness_data = 1834

buffer = 1.1
camera_a_watermark = math.ceil(camera_a_lateness_data / 60) * 60 # Round up to nearest minute
camera_b_watermark = math.ceil(camera_b_lateness_data * buffer) # Add buffer to observed lateness
camera_c_watermark = math.ceil(camera_c_lateness_data * buffer) # Add buffer to observed lateness

print(f"Camera A Watermark: {camera_a_watermark} seconds (rounded up to nearest minute)")
print(f"Camera B Watermark: {camera_b_watermark} seconds (+10% buffer)")
print(f"Camera C Watermark: {camera_c_watermark} seconds (+10% buffer)")

Camera A Watermark: 60 seconds (rounded up to nearest minute)
Camera B Watermark: 712 seconds (+10% buffer)
Camera C Watermark: 2018 seconds (+10% buffer)


## Task 2.1.2 Stream Ingestion
Each Kafka topic (camera-events-A/B/C) will be mapped to one producer, its events are then consumed as JSON and parsed against a fixed schema, while also being watermarked to bound the join window.

Event Schema:

| Field | Type | Description |
|---|---|---|
| `event_id` | String | Unique identifier for the camera event |
| `batch_id` | Integer | Producer batch sequence number |
| `car_plate` | String | Vehicle licence plate |
| `camera_id` | Integer | Camera that recorded the event |
| `timestamp` | String | ISO timestamp of the recording |
| `speed_reading` | Double | Recorded speed in km/h |

### Watermarking
Each stream has independent watermarks which have been precalculated in the previous cell, detailed explanations have been provided on the cell itself to justify why such a watermark was chosen and why its safe.

In [4]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer, watermark_time):
    """
    Reads and parses a Kafka stream for a given topic.
    
    Configures Spark Structured Streaming to read from the specified kafka topic, parsing the
    incoming JSON data from a given schema and applying a watermark based on the given data. Adds metadata
    based on the producer.
    
    Args:
        topic (str): The Kafka topic to subscribe to for this camera stream.
        producer (str): An identifier for the producer/source of the data
        watermark_time (int): Maximum allowed event lateness in seconds
    Returns:
        DataFrame: A Spark DataFrame representing the parsed and enriched stream of camera events.
    """
    
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"kafka:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", f"{watermark_time} seconds")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1", camera_a_watermark)
camera_stream_b = read_camera_stream("camera-events-B", "2", camera_b_watermark)
camera_stream_c = read_camera_stream("camera-events-C", "3", camera_c_watermark)

print("Debug: Kafka streams have been created for all three cameras.")

Debug: Kafka streams have been created for all three cameras.


## Stream Enrichment
Each stream is joined with the static data of camera.csv to enrich the stream with attributes such as `speed_limit`, `position`, and GPS coordinates (`latitude`, `longitude`) which are needed for violation detection and also distance calculation using Haversine.

This stream-static join is used rather than stream-stream join since camera metadata is fixed and doesn't actually change. So we can simply load it as a DataFrame and join them with our camera events stream.

In [5]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit", "latitude", "longitude")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

# Use pandas for preprocessing since we need row-by-row iteration
camera_pd = camera_df.toPandas().sort_values("camera_id").reset_index(drop=True)
camera_pd["camera_id"] = camera_pd["camera_id"].astype(int)

# Calculate max travel time between adjacent cameras
# position is in km, speed_limit in km/h, result in seconds
camera_times = {}
for i in range(1, len(camera_pd)):
    prev_camera = str(int(camera_pd.iloc[i - 1]["camera_id"]))
    curr_camera = str(int(camera_pd.iloc[i]["camera_id"]))
    distance = camera_pd.iloc[i]["position"] - camera_pd.iloc[i - 1]["position"]
    speed_limit = camera_pd.iloc[i]["speed_limit"]
    camera_times[prev_camera, curr_camera] = round(distance / speed_limit * 3600, 9)

print(f"Camera segment travel times (seconds): {camera_times}")

camera_ids = camera_pd["camera_id"].astype(str).tolist()

# Round down with int for some leniency, since we do checking still and not just based on join condition
max_travel_ab = camera_times[(camera_ids[0], camera_ids[1])]
max_travel_bc = camera_times[(camera_ids[1], camera_ids[2])]

print(f"Max travel time between camera 1 and 2: {max_travel_ab} seconds")
print(f"Max travel time between camera 2 and 3: {max_travel_bc} seconds")

def join_stream_with_camera(stream):
    """
    Performs a stream-static join between the Kafka event stream and camera metadata.
    
    Used to enrich the incoming event with position, speed limit, and location data from the camera metadata.
    Inner join is used to discard events from unrecognized camera IDs.
    
    Args:
        stream (DataFrame): The input stream of camera events to be enriched with camera metadata.
    Returns:
        DataFrame: A new DataFrame resulting from the inner join of the event stream with the
    """
    
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

+---------+--------+-----------+-----------+-----------+
|camera_id|position|speed_limit|   latitude|  longitude|
+---------+--------+-----------+-----------+-----------+
|        1|   152.5|        110|2.157730731|102.6601002|
|        2|   153.5|        110|2.162418757|102.6524549|
|        3|   154.5|         90|2.167352891|102.6449144|
+---------+--------+-----------+-----------+-----------+

Debug: Camera loaded: 3 cameras.
Camera segment travel times (seconds): {('1', '2'): 32.727272727, ('2', '3'): 40.0}
Max travel time between camera 1 and 2: 32.727272727 seconds
Max travel time between camera 2 and 3: 40.0 seconds


## Task 2.1.2 Average Speed Violation Detection: Segment Joins

Average speed violations are detected by joining entry and exit events for the same vehicle across different segments of the road. Since the road and camera placement is strictly in the order A -> B -> C, two segment joins are performed: A->B and B->C.

### Join Strategy
We implement an online streaming join across all three topics. Due to the sequential nature of road segments, we join A -> B and B -> C as independent streams. This avoids exponential state explosion while preserving correctness. Spark's global watermark ensures no cross-stream pairs are prematurely evicted.

Segment joins use physical time-ordering rather than `batch_id`, since `batch_id` is a producer-side sequence number and is not synchronised across producers. A higher `batch_id` in Producer B does not necessarily guarantee a later `event_time` than Producer A, so using `batch_id` as a join key would potentially drop valid pairs.

Instead, events are matched on `car_plate` with two time-ordering constraints:
1. `exit.event_time > entry.event_time` - ensures the vehicle passes the exit camera **after** the entry camera, since a vehicle cannot travel in reverse.
2. `exit.event_time <= entry.event_time + max_travel_ab/bc` - only retains pairs where the travel time is lesser than the maximum time a vehicle travelling at exactly the speed limit would take. This means that vehicles travelling at or below the speed limit will fall outside the window and never joined, which is correct since they will not violate the speed limit. This also ensures that we will capture all violating vehicles.


`max_travel_ab` and `max_travel_bc` are derived directly from the camera metadata (segment distance and speed limit), ensuring the join window is data-driven rather than an arbitrary constant.



### Dropped Pairs
When no matching exit event arrives for a given entry within the join window, the entry record is eventually evicted from state by the watermark and logged through the logger function. Each stream has its own independently calculated watermark duration based on the maximum observed out-of-order lateness in its respective CSV. Spark computes a global watermark as the **minimum watermark threshold** across all joined streams, taking the stream whose threshold is furthest behind in time (the slowest stream). This means the slowest stream protects all other streams, ensuring no valid pairs are dropped due to one stream advancing faster than another. An unmatched entry event is dropped once its `event_time` falls below the global watermark threshold, at which point Spark considers it impossible for a valid matching exit event to ever arrive.

In [6]:
def segment_join(entry_stream, exit_stream, max_travel_seconds):
    """
    Joins two camera streams to create a segment between entry and exit points
    
    Args:
        entry_stream (DataFrame): The stream representing the entry camera events.
        exit_stream (DataFrame): The stream representing the exit camera events.
        max_travel_time (int): The maximum allowed travel time between the two cameras in seconds.
    
    Returns:
        DataFrame: A nmew DataFrame representing the segment join between the two cameras
    """
    return (
        entry_stream.alias("entry")
        .join(
            exit_stream.alias("exit"),
            expr(f"""
                entry.car_plate = exit.car_plate
                AND exit.event_time > entry.event_time
                AND exit.event_time <= entry.event_time + interval {max_travel_seconds} seconds
            """),
            "inner"
        )
        .select(
            col("entry.car_plate").alias("car_plate"),
            col("entry.camera_id").alias("start_camera_id"),
            col("exit.camera_id").alias("end_camera_id"),
            col("entry.batch_id").alias("entry_batch_id"),
            col("exit.batch_id").alias("exit_batch_id"),
            col("entry.event_time").alias("entry_time"),
            col("exit.event_time").alias("exit_time"),
            col("entry.position").alias("entry_position"),
            col("exit.position").alias("exit_position"),
            col("exit.speed_limit").alias("speed_limit"),
            col("exit.source").alias("source"),
            col("entry.latitude").alias("entry_latitude"),
            col("entry.longitude").alias("entry_longitude"),
            col("exit.latitude").alias("exit_latitude"),
            col("exit.longitude").alias("exit_longitude"),
        )
    )

# A -> B segment join (camera 1 to camera 2)
segment_ab = segment_join(joined_stream_a, joined_stream_b, max_travel_ab)

# B -> C segment join (camera 2 to camera 3)
segment_bc = segment_join(joined_stream_b, joined_stream_c, max_travel_bc)

print("Debug: Segment joins have been defined for A -> B and B -> C.")

Debug: Segment joins have been defined for A -> B and B -> C.


## Task 2.1.2 Dropped Pair Logging

To be able to observe unmatched entry events, we implement a left-outer join on the same stream pairs for road segments (A->B, B->C) using the same join condition as the violation detection queries. Unlike the inner join used for violation detection, a left-outer join will emit every entry-side row regardless of whether a matching exit event was found, which will fill the exit-side columns with 'null' when no match exists. it is important to note that Spark will not emit the null-padded row immediately when an entry arrives without a match, it will first hold the entry in state and waits. This is important since a valid exit event could still arrive late. So, spark will only emit the row once the watermark has advanced past the point where any fuiture exit event could satisfy the join condition, considering such records permanently unmatched. Only after then will it evict the record from state and emits it exactly once with nulls on the exit side.

Filtering for `exit.car_plate IS NULL` then isolates all entry events that left the join without a match. In Spark Structured Streaming, a left-outer join only emits a null-padded row once the watermark has advanced past the point where a valid match could still arrive, meaning Spark has determined that no future exit event can satisfy the join condition for that entry. At that point the entry is evicted from state and emitted here.

### What gets logged

It is important to note that this query logs **all unmatched entry events**, not exclusively watermark-related data losses. The three categories of records that will appear are:

| Category | Description |
|---|---|
| **Normal non-violations** | Vehicle travelled slower than the speed limit and fell outside the join window — the majority of records |
| **No exit event** | Vehicle never appeared at the downstream camera (e.g. turned off the road) |
| **True late arrivals** | A valid exit event existed but arrived after the watermark threshold and was evicted before the join could complete |


In [7]:
from pyspark.sql.functions import col, lit, isnull

def log_drops_with_reasons(batch_df, batch_id, segment_name):
    """
    Logs expired or unmatched entry/exit event pairs to the console for monitoring.
    
    Called in a foreachBatch on a left-outer streaming join, this function will
    identify records where the watermark advanced past the join window without a matching
    exit event, printing out the car plate, entry time, and reason for drop (e.g. "EXPIRED_WATERMARK") for each unmatched record.
    
    Args:
        batch_idf: Batch DataFrame containing the expired/dropped records
        batch_id: Spark micro-batch identifier
        segment_name: Name of the road segment (for printing purposes)
    Returns:
        None
    """
    if batch_df.isEmpty():
        return
    
    # Convert to Pandas for clean console logging
    batch_df = batch_df.withColumn("entry_time", col("entry_time").cast("string"))
    pdf = batch_df.toPandas()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    print(f"\n[{now}] [{segment_name}] Batch {batch_id}: {len(pdf)} DROPPED/EXPIRED pair(s)")
    
    print("="*60)
    for _, row in pdf.iterrows():
        car = row['car_plate']
        entry_t = str(row.get('entry_time', 'N/A'))
        reason = row.get('drop_reason', 'UNKNOWN')
        details = row.get('details', '')
        print(f"   • {car} | Reason: {reason} | Entry: {entry_t} | {details}")
    print("="*60)

ab_drops_query = (
    joined_stream_a.alias("entry")
    .join(
        joined_stream_b.alias("exit"),
        expr(f"entry.car_plate = exit.car_plate AND exit.event_time > entry.event_time AND exit.event_time <= entry.event_time + interval {max_travel_ab} seconds"),
        "left_outer"
    )
    # Keep only rows where the exit event NEVER arrived or was filtered by join conditions
    .filter(isnull(col("exit.car_plate")))
    .select(
        col("entry.car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("entry.event_time").alias("entry_time"),
        lit("EXPIRED_WATERMARK").alias("drop_reason"),
        lit(f"No valid match within {max_travel_ab}s window").alias("details")
    )
    .writeStream
    .outputMode("append")
    .foreachBatch(lambda df, batch_id: log_drops_with_reasons(df, batch_id, "Segment A→B Drops"))
    .start()
)

bc_drops_query = (
    joined_stream_b.alias("entry")
    .join(
        joined_stream_c.alias("exit"),
        expr(f"entry.car_plate = exit.car_plate AND exit.event_time > entry.event_time AND exit.event_time <= entry.event_time + interval {max_travel_bc} seconds"),
        "left_outer"
    )
    # Keep only rows where the exit event NEVER arrived or was filtered by join conditions
    .filter(isnull(col("exit.car_plate")))
    .select(
        col("entry.car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("entry.event_time").alias("entry_time"),
        lit("EXPIRED_WATERMARK").alias("drop_reason"),
        lit(f"No valid match within {max_travel_bc}s window").alias("details")
    )
    .writeStream
    .outputMode("append")
    .foreachBatch(lambda df, batch_id: log_drops_with_reasons(df, batch_id, "Segment B→C Drops"))
    .start()
)

print("Debug: Drop logging streams started. Unmatched pairs will be printed when their watermark expires.")

Debug: Drop logging streams started. Unmatched pairs will be printed when their watermark expires.


## Task 2.1.3 MongoDB Sink

Violations are persisted to the `violations` collection in MongoDB using `foreachBatch` with pymongo `bulk_write` and `UpdateOne` upserts.

### Daily Merging (Task 2.1.4)

Multiple violations for the same vehicle on the same day are merged into a single document. The upsert match key is `(car_plate, date)`, meaning one document per car per day. Each new violation is added to a `violations` array within that document via `$addToSet`, so duplicates are avoided.

### Retry Handling

Write failures are retried up to **3 times** with a **2-second delay** between attempts. If all retries are exhausted, the batch is logged as dropped rather than crashing the stream.

### Bulk Writes

All operations within a micro-batch are collected into a single `bulk_write` call with `ordered=False`, which maximises write throughput by allowing MongoDB to execute operations in parallel and not halting on a single failure.

### Indexes

The `violations` collection uses a compound index on `(car_plate, date)` which directly matches the upsert filter key, ensuring O(log n) lookups rather than full collection scans on every write. A secondary index on `date` alone supports time-range queries used in visualisation. See `mongo_setup.py` for index creation.

In [8]:

def mongo_sink(name):
    """
    Collects micro-batch violation records, formats them into MongoDB UpdateOne operations,
    and then executes an idempotent bulk upserts with retry logic. The function
    will group violations by (car_plate, date) and uses $addToSet to prevent duplicates within
    the daily violation array.
    
    Args:
        name (str): An identifier for the event source
    Returns:
        function: A function that can be used in foreachBatch to write violations to MongoDB
    """
    def write_violations_to_mongo(batch_df, batch_id):
        """
        Writes the given batch of violation records to MongoDB with retry logic.
        
        Args:
            batch_df (DataFrame): The Spark DataFrame containing the violation records for this batch.
            batch_id: The identifier for the Spark micro-batch (for logging purposes)
        Returns:
            None
        """
        rows = batch_df.collect()
        
        if not rows:
            print(f"[Batch {batch_id}] [{name}]: EMPTY: no violations to write "
                f"(either no matched pairs, or all pairs within speed limit).")
            return

        operations = []
        for row in rows:
            doc = row.asDict()

            # Build the sub-document to push into the violations array
            if doc["violation_type"] == "instantaneous":
                violation_entry = {
                    "type":      "instant",
                    "camera_id": doc["camera_id"],
                    "speed":     doc["speed_recorded"],
                }
            else:
                violation_entry = {
                    "type":         "average",
                    "start_camera": doc["start_camera_id"],
                    "end_camera":   doc["end_camera_id"],
                    "avg_speed":    doc["average_speed"],
                }

            operations.append(
                UpdateOne(
                    {
                        "car_plate": doc["car_plate"],
                        "date": datetime.combine(doc["violation_date"], datetime.min.time()),
                    },
                    {
                        "$addToSet": {"violations": {"$each": [violation_entry]}},
                    },
                    upsert=True
                )
            )
        
        MAX_RETRIES = 3
        RETRY_DELAY = 2  # seconds
        
        client = MongoClient(MONGO_URI)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                collection = client[MONGO_DB]["violations"]
                result = collection.bulk_write(operations, ordered=False)
                print(
                    f"[Batch {batch_id}] [{name}]: {len(operations)} Processed Violations — "
                    f"New Violating Vehicle: {result.upserted_count}, Appended Violations: {result.modified_count}"
                )
                break  # success, exit retry loop
            except Exception as exc:
                print(f"[Batch {batch_id}] [{name}] Attempt {attempt}/{MAX_RETRIES} failed: {exc}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"[Batch {batch_id}] [{name}] All retries exhausted, batch dropped.")
            finally:
                client.close()
    return write_violations_to_mongo

print("Debug: MongoDB sink function defined.")


Debug: MongoDB sink function defined.


## Task 2.1.4 Instantaneous Speed Violation Detection

A vehicle is flagged for an instantaneous violation when its `speed_reading` at the recording camera exceeds that camera's `speed_limit`. This check is applied independently to each of the three streams.

Each violation record retains `event_id` for traceability, and `violation_date` (derived from `event_time`) to support the daily merging logic in MongoDB.

In [ ]:
def get_instant_violations(stream):
    """
    Filters a camera event stream to detect instantaneous speed violations
    
    Compares the vehicle's recorded speed against the static speed limit of the recording camera,
    flagging if a violation occurs and appending metadata. This function will only project
    the relevant fields for instantaneous violations.
    
    Args:
        stream (DataFrame): The input stream of camera events to be checked for instantaneous violations.
    Returns:
        DataFrame: A new DataFrame containing only the records of instantaneous violations with relevant metadata.
    """
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "event_id",
            "car_plate",
            "batch_id",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

def write_json_per_batch(base_path):
    def _writer(batch_df, batch_id):
        file_path = base_path
        if not file_path.endswith(".json"):
            file_path = f"{base_path}/results.json"
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        rows = batch_df.toJSON().collect()
        if not rows:
            return
        with open(file_path, "a", encoding="utf-8") as f:
            for row in rows:
                f.write(row + "\n")
    return _writer

instant_outputs_dir = Path("..") / "outputs" / "instant_violations_json"
for name in ("camera_a", "camera_b", "camera_c"):
    path = instant_outputs_dir / name
    if path.exists():
        shutil.rmtree(path)

camera_a_instant_query = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_a"))
    .start()
)

camera_b_instant_query = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_b"))
    .start()
)

camera_c_instant_query = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_c"))
    .start()
)


print("Debug: Instantaneous violations have been extracted and combined.")


Debug: Instantaneous violations have been extracted and combined.


## Task 2.1.4 Average Speed Violation Detection: Computation

For each matched entry/exit pair from the segment joins, the average speed across the segment is computed as:
```
average_speed (km/h) = distance_km / travel_time_hours
```

**Distance** can be calculated by using either the Haversine formula using the given `latitude` and `longitude` or using the `position` column. Based on the teaching team feedback, it is advised for now that we use the `position` column to calculate our distance between cameras so that it is a round number.

**Travel time** is derived by casting both `entry_time` and `exit_time` to doubles, taking their difference, and dividing by 3600 to convert to hours.

A pair is flagged as a violation only when `average_speed > speed_limit` of the 
**exit camera**, consistent with the AWAS point-to-point enforcement model. The `violation_date` is derived from the `exit_time`, since the exit event is when the violation is confirmed.

In [10]:


def compute_avg_speed(joined_segments):
    """
    Calculates average travel speed across camera segments and filters for violations
    
    Compute distance using the 'position' column provided and travel time using the entry and
    exit timestamps. Flags records where the average speed exceeds the speed limit.
    
    Args:
        joined_segments (DataFrame): The input DataFrame resulting from the segment join between two cameras, 
        containing entry and exit events with their metadata.
    Returns:
        DataFrame: A new DataFrame containing only the records of average speed violations with relevant metadata
    """
    return (
        joined_segments
        .withColumn(
            "distance_km",
            col("exit_position") - col("entry_position")
        )
        .withColumn(
            "travel_time_hours",
            (col("exit_time").cast("double") - col("entry_time").cast("double")) / 3600
                    )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "end_camera_id",
            "average_speed",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "start_camera_id",
            "distance_km",
            "source",
            "entry_time",
            "exit_time",
            "entry_batch_id",
            "exit_batch_id"
    )
)

average_violations_ab = compute_avg_speed(segment_ab)
average_violations_bc = compute_avg_speed(segment_bc)

print("Average speed violation detection logic defined.")

Average speed violation detection logic defined.


## Starting All Streaming Queries

Each violation type and camera combination is wired to a separate `writeStream` query targeting the MongoDB sink. Running them as independent queries allows Spark to manage their trigger schedules and checkpoints separately.

| Query | Source | Violation Type |
|---|---|---|
| `camera_a_query_mongo` | Stream A | Instantaneous |
| `camera_b_query_mongo` | Stream B | Instantaneous |
| `camera_c_query_mongo` | Stream C | Instantaneous |
| `ab_avg_query_mongo` | Segment A→B | Average speed |
| `bc_avg_query_mongo` | Segment B→C | Average speed |
| `camera_speed_query` | Streams A/B/C | Average speed stats |
| `violation_stats_query` | Instant/avg violations | Violation counts |

In [ ]:
camera_a_query_mongo = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_a_instant"))
    .start()
    )

camera_b_query_mongo = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_b_instant"))
    .start()
    )

camera_c_query_mongo = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_c_instant"))
    .start()
    )

ab_avg_query_mongo = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_ab"))
    .start()
    )

bc_avg_query_mongo = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_bc"))
    .start()
    )

print("Debug: MongoDB streaming queries have been started for all violation types.")

Debug: MongoDB streaming queries have been started for all violation types.


In [13]:
# Re-run this cell to refresh the map during the simulation.
# render_camera_map()

[Batch 0] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 0] [camera_a_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 0] [camera_c_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 1] [camera_b_instant]: 3 Processed Violations — New Violating Vehicle: 3, Appended Violations: 0
[Batch 1] [camera_a_instant]: 10 Processed Violations — New Violating Vehicle: 10, Appended Violations: 0
[Batch 0] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 0] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 1] [camera_c_instant]: 1 Processed Violations — New Violating Vehicle: 1, Appended Violations: 0
[Batch 2] [camera_a_instant]: 3 Processed Violations — New Violating Vehicle: 3, Appended Violations: 0
[B

[Batch 17] [camera_a_instant]: 11 Processed Violations — New Violating Vehicle: 11, Appended Violations: 0
[Batch 15] [camera_c_instant]: 1 Processed Violations — New Violating Vehicle: 1, Appended Violations: 0
[Batch 14] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 14] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 16] [camera_b_instant]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 18] [camera_a_instant]: 13 Processed Violations — New Violating Vehicle: 13, Appended Violations: 0
[Batch 16] [camera_c_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 15] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 15] [avg_ab]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 17] [camera_b_instan

[Batch 29] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 30] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 33] [camera_a_instant]: 7 Processed Violations — New Violating Vehicle: 7, Appended Violations: 0
[Batch 30] [camera_c_instant]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 30] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 30] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 34] [camera_a_instant]: 8 Processed Violations — New Violating Vehicle: 8, Appended Violations: 0
[Batch 31] [camera_b_instant]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 31] [camera_c_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batc

[Batch 45] [camera_a_instant]: 9 Processed Violations — New Violating Vehicle: 9, Appended Violations: 0
[Batch 41] [camera_c_instant]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 42] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 43] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 42] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 46] [camera_a_instant]: 11 Processed Violations — New Violating Vehicle: 11, Appended Violations: 0
[Batch 43] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 42] [camera_c_instant]: 1 Processed Violations — New Violating Vehicle: 1, Appended Violations: 0
[Batch 44] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 44] [a

[Batch 53] [camera_b_instant]: 6 Processed Violations — New Violating Vehicle: 0, Appended Violations: 6
[Batch 56] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 57] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 59] [camera_a_instant]: 7 Processed Violations — New Violating Vehicle: 7, Appended Violations: 0
[Batch 53] [camera_c_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).

[2026-05-25 12:53:06] [Segment A→B Drops] Batch 40: 11 DROPPED/EXPIRED pair(s)
   • KC 82 | Reason: EXPIRED_WATERMARK | Entry: 2024-01-01 08:19:46 | No valid match within 32.727272727s window
   • WOQ 4 | Reason: EXPIRED_WATERMARK | Entry: 2024-01-01 08:19:44 | No valid match within 32.727272727s window
   • QK 270 | Reason: EXPIRED_WATERMARK | Entry: 2024-01-01 08:19:45 | No valid match within 32.727272727s window
   • VLY 61 | Reason: EXPIR

...

[Batch 95] [camera_a_instant]: 12 Processed Violations — New Violating Vehicle: 12, Appended Violations: 0
[Batch 96] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 86] [camera_c_instant]: 1 Processed Violations — New Violating Vehicle: 0, Appended Violations: 1
[Batch 96] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 88] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 96] [camera_a_instant]: 9 Processed Violations — New Violating Vehicle: 9, Appended Violations: 0
[Batch 97] [avg_bc]: 1 Processed Violations — New Violating Vehicle: 0, Appended Violations: 1
[Batch 87] [camera_c_instant]: 1 Processed Violations — New Violating Vehicle: 1, Appended Violations: 0
[Batch 97] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 98] [avg_bc]: 1 Proces

[Batch 99] [camera_b_instant]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 108] [camera_a_instant]: 9 Processed Violations — New Violating Vehicle: 9, Appended Violations: 0
[Batch 98] [camera_c_instant]: 2 Processed Violations — New Violating Vehicle: 1, Appended Violations: 1
[Batch 110] [avg_bc]: 1 Processed Violations — New Violating Vehicle: 0, Appended Violations: 1
[Batch 100] [camera_b_instant]: 1 Processed Violations — New Violating Vehicle: 0, Appended Violations: 1
[Batch 110] [avg_ab]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 109] [camera_a_instant]: 10 Processed Violations — New Violating Vehicle: 10, Appended Violations: 0
[Batch 99] [camera_c_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 111] [avg_bc]: 3 Processed Violations — New Violating Vehicle: 0, Appended Violations: 3
[Batch 111] [avg_ab]: EMPTY: no violations to write (either n

[Batch 123] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 120] [camera_a_instant]: 9 Processed Violations — New Violating Vehicle: 9, Appended Violations: 0
[Batch 110] [camera_c_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 124] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 124] [avg_ab]: 3 Processed Violations — New Violating Vehicle: 0, Appended Violations: 3
[Batch 112] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 125] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 121] [camera_a_instant]: 7 Processed Violations — New Violating Vehicle: 7, Appended Violations: 0
[Batch 111] [camera_c_instant]: 1 Processed Violations — New Violating Vehicle: 0, Appended Violations: 1
[Batch

[Batch 122] [camera_b_instant]: 5 Processed Violations — New Violating Vehicle: 0, Appended Violations: 5
[Batch 131] [camera_a_instant]: 12 Processed Violations — New Violating Vehicle: 12, Appended Violations: 0
[Batch 138] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 139] [avg_bc]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 121] [camera_c_instant]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 140] [avg_bc]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 123] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 139] [avg_ab]: 5 Processed Violations — New Violating Vehicle: 0, Appended Violations: 5
[Batch 132] [camera_a_instant]: 10 Processed Violations — New Violating Vehicle: 10, Appended Violations: 0
[Batch 122] [camera_c_instant]: EM

[Batch 142] [camera_a_instant]: 12 Processed Violations — New Violating Vehicle: 12, Appended Violations: 0
[Batch 133] [camera_b_instant]: 3 Processed Violations — New Violating Vehicle: 1, Appended Violations: 2
[Batch 153] [avg_ab]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 154] [avg_bc]: 1 Processed Violations — New Violating Vehicle: 0, Appended Violations: 1
[Batch 143] [camera_a_instant]: 8 Processed Violations — New Violating Vehicle: 8, Appended Violations: 0
[Batch 132] [camera_c_instant]: 2 Processed Violations — New Violating Vehicle: 2, Appended Violations: 0
[Batch 134] [camera_b_instant]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 154] [avg_ab]: EMPTY: no violations to write (either no matched pairs, or all pairs within speed limit).
[Batch 155] [avg_bc]: 2 Processed Violations — New Violating Vehicle: 0, Appended Violations: 2
[Batch 144] [camera_a_instant]: 9 Processed Violatio